# Importando tudo

In [ ]:
# !git clone <URL> ykk_repo
# !git -C ./ykk_repo/ pull

# import sys
# sys.path.append('./ykk_repo/')

import os
os.environ['YKK_NOCHDIR'] = '1'

import ykk
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from IPython.display import clear_output
ykk.urban8k().download_dataset()
clear_output()

def separate_folds(val_fold):
    """Dado um fold de validação, retorna os folds
    de treino(todos que não são de validação)
    """
    return [f for f in range(1, 11) if f != val_fold]

def get_data(val_fold,fs,n_mfcc=13,mel_kw=None):
    """Splits dataset into validation fold and training fold.
    Returns:
        (Dataloader,Dataloader): Training and Validation Fold
    """
    train_fld = separate_folds(val_fold)
    valid_data = ykk.urban8k(augmentation=False,
                            fs=fs,filter_folds=val_fold)

    train_data = ykk.urban8k(augmentation=True,
                            fs=fs,filter_folds=train_fld)

    for _data in [train_data,valid_data]:
        _data.set_mfcc_params(n_mfcc = n_mfcc,
                              mel_kw = mel_kw)

    train_dl = DataLoader(train_data,shuffle=True,)
    valid_dl = DataLoader(valid_data)
    
    return train_dl,valid_dl

## Parâmetros

In [ ]:
fs = 44100
mel_kw = {"n_fft": 2048, "hop_length": 256, "n_mels": 23}
n_mfcc = 13
n_frames = (int(fs*4.0)-mel_kw['n_fft'])/256 + 1

n_features = n_frames*n_mfcc

In [ ]:
class NeuralNetwork(nn.Module):
    def __init__(self,n_features,n_hidden=512,dropout=.5):
        super().__init__()
        self.n_features = n_features
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            # Hidden 1
            nn.Linear(n_features, n_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            # Hidden 2
            nn.Linear(n_hidden, n_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            # Out
            nn.Linear(n_hidden, 10)
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

In [ ]:
valid_fld = 1
train_dl,valid_dl = get_data(valid_fld,fs=fs)

In [ ]:
model = NeuralNetwork(n_features=n_features,
                      dropout=.3
                      )
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), 
                            lr=1e-3
                            )

In [ ]:
n_epochs = 2
trainer = ykk.NetTrainer(model   = model,
                         loss_fn = loss_fn,
                         )

for epoch in range(n_epochs):
    trainer.train(train_dl,
                  optimizer=optimizer)
    trainer.validate(valid_dl)